## If working in Google Colab, run this first
Run the cell below at the start of every session. It downloads the course files to your Google Drive and keeps them up to date.
When you see "Setup complete", you are ready to run the rest of the notebook.

**Do not edit or save your work inside the `AIforMovementScience` folder.**
Use **File > Save a copy in Drive** and work on your own copy.

In [ ]:
# [course-setup] Run this cell first, every session (or use Runtime > Run all)
import os, sys
try:
    from google.colab import drive
    os.chdir('/content')                  # start from a valid directory
    drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/AIforMovementScience'
    if os.path.isdir(REPO + '/.git'):
        # already downloaded before: just update it
        !git -C "$REPO" fetch origin main && git -C "$REPO" reset --hard origin/main
    else:
        # first time, or a broken/partial copy: download a fresh copy
        !rm -rf "$REPO"
        !git clone https://github.com/dguari1/AIforMovementScience.git "$REPO"
except ImportError:
    # not in Colab: walk up from here until the repository root is found
    REPO = os.getcwd()
    while REPO != '/' and not os.path.isdir(os.path.join(REPO, 'lectures', 'data')):
        REPO = os.path.dirname(REPO)

WORKDIR = REPO + '/lectures'
os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)
print('Setup complete. You can now run the rest of the notebook.')

# Pose estimation - Application, part 3: can a video camera replace the gait lab?

We use the feature table of part 2 to answer three questions:

1. Can the **video features** predict the **GDI**? Are they better than the demographics (age, height, mass, BMI)?
2. For **GMFCS** (the target of lecture 04.03), how do the video features compare with the motion-capture measures?
3. Is the video only measuring **walking speed**, or does it give more information?

Rules from module 04 that we use here:
- many patients have more than one video, so we split **by patient** with `GroupKFold`;
- we always compare with a simple **baseline**;
- we report the mean **and the spread** across the folds.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold, cross_val_predict, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, roc_auc_score

features = pd.read_csv('data/cp_video_gait/cp_video_features.csv')
print('Videos with a missing value (height/mass not recorded):', features.isna().any(axis=1).sum(), '-> removed')
features = features.dropna().reset_index(drop=True)
id_cols = ['video_id', 'patient_id', 'exam_id', 'diagnosis']
demo_cols = ['age', 'height', 'mass', 'bmi']
lab_cols = ['speed', 'cadence']
video_cols = [c for c in features.columns if c not in id_cols + ['gmfcs', 'GDI'] + demo_cols + lab_cols]
groups = features['patient_id']
cv = GroupKFold(n_splits=5)
print(f'{len(features)} videos, {groups.nunique()} patients, {len(video_cols)} video features')

## 1. Predict the GDI (regression)

We compare four sets of inputs, each with two models: ridge regression (linear) and a random forest (nonlinear). For each video, `cross_val_predict` gives the prediction of the model that did **not** see this patient during training.

We report:
- **r**: the correlation between the predicted and the true GDI,
- **MAE**: the mean absolute error, in GDI points.

In [ ]:
input_sets = {'Demographics': demo_cols,
              'Video features': video_cols,
              'Video + demographics': video_cols + demo_cols,
              'Motion capture (speed, cadence)': lab_cols}
models = {'Ridge': Pipeline([('scaler', StandardScaler()), ('model', RidgeCV(alphas=np.logspace(-2, 3, 20)))]),
          'Random forest': RandomForestRegressor(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=0)}

y = features['GDI']
rows, predictions = [], {}
for set_name, cols in input_sets.items():
    for model_name, model in models.items():
        pred = cross_val_predict(model, features[cols], y, cv=cv, groups=groups)
        predictions[(set_name, model_name)] = pred
        rows.append({'inputs': set_name, 'model': model_name,
                     'r': np.corrcoef(pred, y)[0, 1], 'MAE': mean_absolute_error(y, pred)})
gdi_results = pd.DataFrame(rows)
print(f'Baseline (always predict the mean GDI): MAE = {mean_absolute_error(y, np.full(len(y), y.mean())):.2f}')
gdi_results.round(3)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
for a, key in zip(ax, [('Demographics', 'Random forest'), ('Video features', 'Random forest')]):
    a.scatter(y, predictions[key], s=5, alpha=0.3)
    a.plot([30, 115], [30, 115], 'k--')
    a.set_xlabel('True GDI (motion capture)'); a.set_title(f'{key[0]} ({key[1]})')
ax[0].set_ylabel('Predicted GDI')
plt.show()

The video features predict GDI much better than the demographics, and better than speed and cadence from the motion-capture laboratory. The random forest is better than the linear model: the relation between the features and GDI is not linear.

The predictions are still far from perfect: the predicted values are pulled toward the mean (high GDI values are under-estimated, low values are over-estimated). The authors of the study used a CNN on the full trajectories and reported r = 0.73. Our simple features reach a similar level.

## Which video features are important?

In [ ]:
rf = RandomForestRegressor(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=0)
rf.fit(features[video_cols], y)
importance = pd.Series(rf.feature_importances_, index=video_cols).sort_values()
importance.tail(10).plot.barh(figsize=(7, 4))
plt.xlabel('Importance (random forest)'); plt.title('Top 10 video features for GDI')
plt.show()

Remember from lecture 04.02: feature importance shows what the model **uses**, not what **causes** the GDI. Features that are correlated share their importance.

## 2. GMFCS I–II vs III–IV (the target of lecture 04.03)

We use only the valid GMFCS levels (1 to 4), as in 04.03, and the AUC.

In [ ]:
gm = features[features.gmfcs.between(1, 4)].copy()
y_gm = (gm.gmfcs >= 3).astype(int)
print(f'{len(gm)} videos, {gm.patient_id.nunique()} patients, {y_gm.mean():.0%} GMFCS III-IV')

clf_sets = {'Demographics': demo_cols,
            'Video features': video_cols,
            'Motion capture speed only': ['speed'],
            'Video features + speed': video_cols + ['speed']}
rows = []
for set_name, cols in clf_sets.items():
    lr = Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=2000, class_weight='balanced'))])
    rfc = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, class_weight='balanced', n_jobs=-1, random_state=0)
    for model_name, model in [('Logistic regression', lr), ('Random forest', rfc)]:
        auc = cross_val_score(model, gm[cols], y_gm, cv=cv, groups=gm.patient_id, scoring='roc_auc')
        rows.append({'inputs': set_name, 'model': model_name, 'AUC mean': auc.mean(), 'AUC SD': auc.std()})
pd.DataFrame(rows).round(3)

In 04.03, the **speed alone** from motion capture gave an AUC of about 0.94. The video features give a similar AUC. With video features + speed, the AUC is a little higher than with speed alone, so the video gives some information that the speed does not.

## 3. Is the video only measuring the speed?

Faster patients usually have a less impaired gait, so speed is a strong predictor (lecture 04.03). Can the video features predict the speed? And do they predict GDI better than the speed does?

In [ ]:
rf = RandomForestRegressor(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=0)
speed_pred = cross_val_predict(rf, features[video_cols], features['speed'], cv=cv, groups=groups)
print(f"Video features -> speed:  r = {np.corrcoef(speed_pred, features['speed'])[0, 1]:.2f}")

gdi_from_speed = cross_val_predict(rf, features[['speed']], y, cv=cv, groups=groups)
gdi_from_video = predictions[('Video features', 'Random forest')]
gdi_from_both = cross_val_predict(rf, features[video_cols + ['speed']], y, cv=cv, groups=groups)
for name, p in [('speed only', gdi_from_speed), ('video features', gdi_from_video), ('video + speed', gdi_from_both)]:
    print(f"GDI from {name:15s}: r = {np.corrcoef(p, y)[0, 1]:.2f}")

The video features contain information about the speed (the step frequency is one of them), but they predict GDI better than the speed alone. So the video also captures **how** the patient walks (knee and hip movement, trunk sway, pelvis tilt), not only **how fast**.

## Summary and limits

- With enough data (more than 3,000 videos), simple features from 2D pose estimation predict a motion-capture gait index (GDI) and the GMFCS level much better than the demographics.
- Compare with part 1: with only 89 videos, the movement measures added little. **The amount of data matters.**

Limits:
- The keypoints are 2D. Angles depend on the camera view (side or front), and the camera moves.
- In each frame, the person with the highest confidence was used: some frames may show another person (a parent or a clinician).
- All videos come from one hospital. A model trained here may not work as well on videos from another clinic, or from a phone at home.
- The frame rate is not known exactly, so time-based measures are estimates.

## Activity

1. Train the random forest for GDI with only the **knee** features (`[c for c in video_cols if 'knee' in c]`). How much of the performance do you keep?
2. Remove `step_freq_per_frame` from the video features. What happens to the GDI prediction? What does this tell you about the role of walking speed?
3. **Extension (module 05):** the trajectories in `cp_video_trajectories.npz` are signals with several channels, like the UCI HAR data. How would you prepare them for a 1D CNN? (Hint: shape, normalization, missing values, split by patient.)